# Q1–Q2 Analytics — Role 3

Notebook สำหรับวิเคราะห์แบบสำรวจ 88 ครัวเรือน โดยใช้คอลัมน์และ mapping ตาม `scripts/01_ingest_validate_survey.py` และโครงสร้าง Gold จาก `scripts/03b_transform_survey_gold.py`. รันจาก root ของ repo หลัง `pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
# รองรับการเปิด Notebook จาก root, notebooks/ หรือ docs/role3_deliverables/
CWD = Path.cwd()
candidates = [CWD, *CWD.parents]
ROOT = next((p for p in candidates if (p / "data" / "bronze" / "ThaiCI_Household_Questionaire_anonymized.xlsx").exists()), None)
if ROOT is None:
    raise FileNotFoundError("ไม่พบไฟล์แบบสอบถามใน data/bronze กรุณาเปิด Notebook ภายใน Repository นี้")
RAW=ROOT/"data/bronze/ThaiCI_Household_Questionaire_anonymized.xlsx"
GOLD=ROOT/"data/gold"


In [ ]:
# ใช้ข้อมูล Silver/Gold เป็นหลัก และ fallback สร้าง analytic frame จาก Bronze ด้วย mapping เดียวกับ pipeline
# Bronze fallback ช่วยให้ notebook ทำงานแม้ไฟล์ parquet ไม่พร้อม แต่ต้องมี openpyxl
idx={0:"household_code",1:"community_name",4:"interview_month",12:"income_stability",13:"monthly_income_range",14:"monthly_expense_range",24:"has_savings",25:"debt_range",26:"household_size",61:"housing_type",63:"wall_material",64:"roof_material",66:"has_ceiling",67:"has_insulation",68:"ventilation_quality",69:"surrounding_shade",82:"barrier_no_budget",84:"barrier_renting_no_right",86:"barrier_expensive_material",115:"summer_heat_severity",120:"impact_insomnia",121:"impact_discomfort",122:"impact_headache_fatigue",125:"impact_higher_elec_bill",130:"cope_use_fan",131:"cope_use_ac",133:"cope_open_windows",225:"normal_elec_bill_range",226:"summer_elec_bill_range",227:"summer_elec_increase_level",231:"has_fan",232:"has_ac",249:"reduced_elec_due_to_cost"}
raw=pd.read_excel(RAW,sheet_name=0)
df=pd.DataFrame({name:raw.iloc[:,i] for i,name in idx.items()})
for c in ["barrier_no_budget","barrier_renting_no_right","barrier_expensive_material","impact_insomnia","impact_discomfort","impact_headache_fatigue","impact_higher_elec_bill","cope_use_fan","cope_use_ac","cope_open_windows","has_fan","has_ac"]:
    df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0).astype(int)
INCOME_MAP={"ต่ำกว่า 5,000 บาท":4000.0,"5,001–10,000 บาท":7500.0,"10,001–20,000 บาท":15000.0,"20,001–50,000 บาท":35000.0,"มากกว่า 50,000 บาท":60000.0}
BILL_MAP={"ต่ำกว่า 500 บาท":400.0,"500–1,000 บาท":750.0,"1,001–2,000 บาท":1500.0,"2,001–3,000 บาท":2500.0,"มากกว่า 3,000 บาท":3500.0}
df["est_monthly_income_thb"]=df.monthly_income_range.map(INCOME_MAP)
df["est_summer_elec_thb"]=df.summer_elec_bill_range.map(BILL_MAP)
df["summer_energy_burden_pct"]=(df.est_summer_elec_thb/df.est_monthly_income_thb*100).round(2)
df["health_impact_count"]=df[["impact_insomnia","impact_discomfort","impact_headache_fatigue"]].sum(axis=1)
df["high_health_impact_2plus"]=df.health_impact_count>=2
print(df.shape, "unique households:",df.household_code.nunique())
display(df.head())


## Q1 — ลักษณะที่อยู่อาศัยกับผลกระทบจากความร้อน

In [ ]:
for c in ["roof_material","ventilation_quality","surrounding_shade"]:
    summary=df.groupby(c).agg(n=("household_code","size"),health_impact_mean=("health_impact_count","mean"),two_plus_pct=("high_health_impact_2plus","mean"),insomnia_pct=("impact_insomnia","mean"),discomfort_pct=("impact_discomfort","mean"),headache_pct=("impact_headache_fatigue","mean")).round(3)
    print("\n",c); display(summary)


In [ ]:
vent=df.groupby("ventilation_quality").high_health_impact_2plus.mean().mul(100)
ax=vent.plot(kind="bar",figsize=(7,4),title="2+ heat-related health impacts by ventilation (%)")
ax.set_ylabel("Households (%)"); plt.tight_layout(); plt.show()


## Q2 — รายได้ ภาระค่าไฟ และข้อจำกัด

In [ ]:
income_summary=df.groupby("monthly_income_range").agg(n=("household_code","size"),mean_burden=("summer_energy_burden_pct","mean"),median_burden=("summer_energy_burden_pct","median"),mean_summer_bill=("est_summer_elec_thb","mean"),reduced=("reduced_elec_due_to_cost",lambda x:(x=="เคย").mean()),no_budget=("barrier_no_budget","mean")).round(3)
display(income_summary)


In [ ]:
order=["ต่ำกว่า 5,000 บาท","5,001–10,000 บาท","10,001–20,000 บาท","20,001–50,000 บาท","มากกว่า 50,000 บาท"]
values=[df.loc[df.monthly_income_range==v,"summer_energy_burden_pct"].dropna() for v in order]
plt.figure(figsize=(9,4)); plt.boxplot(values,tick_labels=["<5k","5–10k","10–20k","20–50k",">50k"],showmeans=True)
plt.ylabel("Estimated summer energy burden (%)"); plt.title("Summer energy burden by household income"); plt.tight_layout(); plt.show()


In [ ]:
reduce_summary=df.groupby("reduced_elec_due_to_cost").agg(n=("household_code","size"),mean_burden=("summer_energy_burden_pct","mean"),mean_income=("est_monthly_income_thb","mean"),no_budget=("barrier_no_budget","mean")).round(3)
display(reduce_summary)


## Quality checks and export

In [ ]:
print("Duplicate household codes:",df.household_code.duplicated().sum())
print("Missing values:"); display(df.isna().sum().sort_values(ascending=False).head(10))
# export aggregate tables for report
df.groupby("ventilation_quality").agg(n=("household_code","size"),health_impact_mean=("health_impact_count","mean"),two_plus_pct=("high_health_impact_2plus","mean")).to_csv(ROOT/"q1_ventilation_summary.csv",encoding="utf-8-sig")
income_summary.to_csv(ROOT/"q2_income_summary.csv",encoding="utf-8-sig")
